In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('100_Unique_QA_Dataset.csv')

In [3]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [15]:
# tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace('?', '')
    text = text.replace('"', '')
    return text.split()

In [16]:
tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [26]:
# vocab
vocab = {'<UNK>': 0}
def build_vocab(row):
    tokenize_question = tokenize(row['question'])
    tokenize_answer = tokenize(row['answer'])

    merged_token = tokenize_question + tokenize_answer

    for token in merged_token:
        if token not in vocab:
            vocab[token] = len(vocab)

In [27]:
df.apply(build_vocab, axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [29]:
len(vocab)

326

In [30]:
# covert words to numerical indices
def text_to_indices(text, vocab):
    indexed_text = []

    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text

In [31]:
text_to_indices("What is campusx", vocab)

[1, 2, 0]

In [33]:
import torch
from torch.utils.data import Dataset, DataLoader

In [34]:
class QADataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab
        
    def __getitem__(self, index):
        numeric_question =  text_to_indices(self.df.iloc[index]['question'], self.vocab)
        numeric_answer =  text_to_indices(self.df.iloc[index]['answer'], self.vocab)

        return torch.tensor(numeric_question), torch.tensor(numeric_answer)
    def __len__(self):
        return self.df.shape[0]

In [35]:
dataset = QADataset(df, vocab)

In [36]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))